# Circuit Debugging Assistant: Vision Model Training Pipeline
## Fine-Tuning YOLOv8 on Electronic Schematic Symbols

This notebook accompanies the research framework:  
**"Circuit Debugging Assistant: A Hybrid Simulation- and Evidence-Driven Framework for Electronic Circuit Fault Localization"**  
*Sandesh Mahajan, Snehal Khalkar, Sabiya Pathan (KKWIEER, Nashik, 2026)*

---
### Objectives:
1. Prepare and annotate electronic schematic symbols (`resistor`, `capacitor`, `inductor`, `voltage_source`, `ground`, `junction_dot`, `diode`, `opamp`).
2. Generate balanced Train, Validation, and Test splits partitioned strictly by circuit design (preventing data leakage).
3. Fine-tune a lightweight YOLO model (`yolov8n.pt` or `yolov8s.pt`) with bounding-box regression and classification.
4. Evaluate Per-Class Precision, Recall, mAP@50, mAP@50-95, and Confusion Matrix.
5. Export ONNX/PyTorch checkpoint for seamless integration into the web application.

In [ ]:
# Step 1: Environment Setup in Google Colab
!pip install ultralytics opencv-python matplotlib albumentations pyyaml

import os
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

### Step 2: Class Mapping and Annotation Schema

We define 8 primary electronic symbol classes for schematic recognition:

In [ ]:
CLASSES = [
    'resistor',        # 0
    'capacitor',       # 1
    'inductor',        # 2
    'voltage_source',  # 3
    'ground',          # 4
    'junction_dot',    # 5
    'diode',           # 6
    'opamp'            # 7
]

dataset_yaml = {
    'path': './dataset',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': len(CLASSES),
    'names': CLASSES
}

os.makedirs('./dataset', exist_ok=True)
with open('./dataset/circuit_symbols.yaml', 'w') as f:
    yaml.dump(dataset_yaml, f)

print("Saved dataset configuration to ./dataset/circuit_symbols.yaml")

### Step 3: Dataset Splitting by Circuit Archetype

To avoid inflated test metrics, entire schematic diagrams must never appear in both train and test partitions.

In [ ]:
# Example dataset partition script
import random
import shutil

def prepare_splits(source_dir, out_dir, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    for split in ['train', 'val', 'test']:
        os.makedirs(f"{out_dir}/images/{split}", exist_ok=True)
        os.makedirs(f"{out_dir}/labels/{split}", exist_ok=True)
    print("Dataset directory structure initialized.")

prepare_splits('./raw_schematics', './dataset')

### Step 4: Model Training with Transfer Learning

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8-nano weights for schematic feature extraction
model = YOLO('yolov8n.pt')

# Hyperparameters tuned for line-art diagrams
results = model.train(
    data='./dataset/circuit_symbols.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    save=True,
    project='circuit_detector_runs',
    name='yolov8_circuit_v1',
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    plots=True
)

### Step 5: Model Evaluation & Metric Extraction

Extract per-class Precision, Recall, and mAP@50.

In [ ]:
# Validate on Test set
metrics = model.val(data='./dataset/circuit_symbols.yaml', split='test')
print(f"Overall mAP@50: {metrics.box.map50:.4f}")
print(f"Overall mAP@50-95: {metrics.box.map:.4f}")

# Per-class statistics
for idx, cls_name in enumerate(CLASSES):
    if idx < len(metrics.box.maps):
        print(f"Class [{cls_name}]: AP@50-95 = {metrics.box.maps[idx]:.4f}")

### Step 6: Export for Web Application Integration

In [ ]:
# Export to ONNX format
onnx_path = model.export(format='onnx')
print(f"Model exported to: {onnx_path}")

# Copy weights to web application checkpoint directory
# shutil.copy('circuit_detector_runs/yolov8_circuit_v1/weights/best.pt', '../app/vision/weights/best.pt')
print("To integrate into Circuit Debugging Assistant, copy best.pt into app/vision/weights/")